# Run survival pipeline locally

Same commands as `bash_scripts/launch_survival.sh` and `array_survival_run.sh`, just invoked from a notebook so you can iterate on one model/landmark/config at a time without SLURM.

**Before running:** make sure the active kernel is the `survlatent_ode` conda env (or activate it inline with `conda run -n survlatent_ode python …`). Edit the `Paths` cell to point at your data.

## Paths

Defaults are the cluster paths from `bash_scripts/build_prediction_inputs.sh`. Override `PROJECT_ROOT`, `DATA`, `V3_LABELS`, `INPUTS_DIR`, `OUTPUT_DIR` for local copies.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
DATA         = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/longitudinal_prediction_data.csv")
V3_LABELS    = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/v3_outputs/LLM_v3_labels.tsv")
INPUTS_DIR   = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis/prediction_inputs")
OUTPUT_DIR   = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis/local_runs")

SURVIVAL_DIR = PROJECT_ROOT / "COMPASS" / "survival_analysis"

os.chdir(PROJECT_ROOT)
INPUTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# BLAS thread caps (mirror the bash scripts so models don't oversubscribe cores).
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

print("cwd:        ", os.getcwd())
print("inputs_dir: ", INPUTS_DIR)
print("output_dir: ", OUTPUT_DIR)

## 1. Build prediction inputs

Required after any change to `build_prediction_inputs.py` (e.g. the `landmark_day` fix). Writes `aggregated_landmark{D}.csv`, `longitudinal_landmark{D}.csv`, etc. into `INPUTS_DIR`.

In [ ]:
!python {SURVIVAL_DIR}/build_prediction_inputs.py \
    --data {DATA} \
    --v3-labels-path {V3_LABELS} \
    --output-dir {INPUTS_DIR} \
    --landmark-days 0 90 \
    --time-unit-days 7 \
    --test-frac 0.20 \
    --val-frac 0.20 \
    --min-patient-coverage 0.20

## 2. Choose one task to run

Set `MODEL`, `LANDMARK`, `CONFIG`. Valid combinations (from `slurm_manifests/survival_tasks.tsv`):

| MODEL | LANDMARK | CONFIG |
|---|---|---|
| `cox` | `0` or `90` | `both` (or `univariate` / `multivariable`) |
| `xgboost` | `0` or `90` | _(ignored)_ |
| `deephit` | `0` or `90` | `competing` / `PLATINUM` / `DEATH` |
| `cox_genomic` | _(ignored)_ | _(ignored)_ |

In [ ]:
MODEL    = "deephit"
LANDMARK = 0
CONFIG   = "PLATINUM"
N_FOLDS  = 5

ROW_OUTPUT_DIR = OUTPUT_DIR / MODEL / f"landmark_{LANDMARK}" / CONFIG
ROW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Writing to:", ROW_OUTPUT_DIR)

### Cox

In [ ]:
!python {SURVIVAL_DIR}/cox_aggregated.py \
    --inputs-dir {INPUTS_DIR} \
    --output-dir {ROW_OUTPUT_DIR} \
    --landmark-days {LANDMARK} \
    --analysis {CONFIG} \
    --endpoints platinum death \
    --n-folds {N_FOLDS}

### XGBoost

In [ ]:
!python {SURVIVAL_DIR}/landmark_xgboost.py \
    --inputs-dir {INPUTS_DIR} \
    --output-dir {ROW_OUTPUT_DIR} \
    --landmark-days {LANDMARK} \
    --endpoints platinum death \
    --n-folds {N_FOLDS}

### Dynamic DeepHit

Add `--cuda` if a GPU is available. For a fast smoke test, shrink the CV grid: `--cv-hidden-dims 64 --cv-dropouts 0.20 --cv-lrs 1e-3 --n-folds 2`.

In [ ]:
!python {SURVIVAL_DIR}/dynamic_deephit.py \
    --inputs-dir {INPUTS_DIR} \
    --output-dir {ROW_OUTPUT_DIR} \
    --landmark-day {LANDMARK} \
    --config {CONFIG} \
    --n-folds {N_FOLDS}

### Cox genomic (univariate)

Anchor is `t_sample`, not a landmark day.

In [ ]:
GENOMIC_OUTPUT_DIR = OUTPUT_DIR / "cox_genomic" / "univariate"
GENOMIC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

!python {SURVIVAL_DIR}/cox_genomic_univariate.py \
    --inputs-dir {INPUTS_DIR} \
    --output-dir {GENOMIC_OUTPUT_DIR} \
    --endpoints platinum death

## 3. Inspect outputs

In [ ]:
import pandas as pd

for csv in sorted(ROW_OUTPUT_DIR.glob("*.csv")):
    print(csv.name)

# Example: load deephit metrics
metrics_path = ROW_OUTPUT_DIR / f"dynamic_deephit_metrics_{CONFIG}.csv"
if metrics_path.exists():
    display(pd.read_csv(metrics_path))